In [1]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
app_id = os.getenv("INSTAGRAM_APP_ID")
app_secret = os.getenv("INSTAGRAM_APP_SECRET")
redirect_uri = os.getenv("REDIRECT_URL")
instagram_business_account_id = os.getenv("INSTAGRAM_BUSINESS_ACCOUNT_ID")
long_access_token = os.getenv("INSTAGRAM_LONG_ACCESS_TOKEN")

# Get Authorization Code

## Open in Browser Only

In [3]:
url = "https://www.instagram.com/oauth/authorize?"
url = url + f"client_id={int(app_id)}"
url = url + "&" + f"redirect_uri={redirect_uri}"
url = url + "&" + f"response_type=code"
url = url + "&" + f"scope={('instagram_business_basic,instagram_business_content_publish,instagram_business_manage_messages,instagram_business_manage_comments').replace(',','%2C')}"
url

'https://www.instagram.com/oauth/authorize?client_id=1513001943743124&redirect_uri=https://freezingly-frigid-francoise.ngrok-free.dev/&response_type=code&scope=instagram_business_basic%2Cinstagram_business_content_publish%2Cinstagram_business_manage_messages%2Cinstagram_business_manage_comments'

## Paste url from browser and get authorization_code

In [4]:
returned_url = "https://freezingly-frigid-francoise.ngrok-free.dev/?code=AQDd7dmB3ElKrFhKiht8q_nFirVT3MYcyXQdBm-RGW4GtAu_MrvWOWw3XwvXXYjrRAbOqXjyEuRmeWuHqT5zChosdVySGUIFs5PG1pq3iCB_k0uZeUcHKoHF5reDhU6zcsZKCewiHWTuuFAgvsIy5aQPw7E9mvQ25P5KZtNFMQvW1s0D7u5rp5jvTk9KfvYZusDeh5x3s-p5vyVddnos3AolWwEM7WgQP_jNU2pN6HKvLg#_"
authorization_code = returned_url.replace(redirect_uri + "?code=", "")
authorization_code

'AQDd7dmB3ElKrFhKiht8q_nFirVT3MYcyXQdBm-RGW4GtAu_MrvWOWw3XwvXXYjrRAbOqXjyEuRmeWuHqT5zChosdVySGUIFs5PG1pq3iCB_k0uZeUcHKoHF5reDhU6zcsZKCewiHWTuuFAgvsIy5aQPw7E9mvQ25P5KZtNFMQvW1s0D7u5rp5jvTk9KfvYZusDeh5x3s-p5vyVddnos3AolWwEM7WgQP_jNU2pN6HKvLg#_'

# Get User Access Token

In [5]:
# O access_token serve para pegarmos o long_access_token, se ele já existe não faz sentido o pegarmos
if not long_access_token:
    url = f"https://api.instagram.com/oauth/access_token"
    form_data = {
        "client_id": int(app_id),
        "client_secret": app_secret,
        "grant_type": "authorization_code",
        "redirect_uri": redirect_uri,
        "code": authorization_code,
    }
    response = requests.post(url, data=form_data)
    data = response.json()
    print(json.dumps(data, indent=4))
    user_access_token = data["access_token"]
    print("Your user token is: " + user_access_token[0:5] + "...")

# Long Access Token

In [6]:
# Caso não existe um long_access_token configurado no .env, o pega aqui
if not long_access_token:
    url = f"https://graph.instagram.com/access_token"
    payload = {
        "client_secret": app_secret,
        "grant_type": "ig_exchange_token",
        "access_token": user_access_token
    }
    response = requests.get(url, params=payload)
    data = response.json()
    print(json.dumps(data, indent=4))
    long_access_token = data["access_token"]
    print("Your user token is: " + long_access_token[0:5] + "...")

# Test API

In [7]:
url = f"https://graph.instagram.com/v25.0/me"
payload = {
    "fields": "id,user_id,username,name,account_type,profile_picture_url,followers_count,follows_count,media_count",
    "access_token": long_access_token
}
response = requests.get(url, params=payload)
data = response.json()
print(json.dumps(data, indent=4))
print(f"App Scoped User ID: {data['id']}")
print(f"Instagram Business Account User ID: {data['user_id']}")

{
    "id": "26139958472366508",
    "user_id": "17841400550881709",
    "username": "eduufiuza",
    "name": "Eduardo Fi\u00faza Cruz",
    "account_type": "MEDIA_CREATOR",
    "profile_picture_url": "https://scontent.cdninstagram.com/v/t51.2885-19/135318839_231318785235031_3852435900524642357_n.jpg?stp=dst-jpg_s206x206_tt6&_nc_cat=101&ccb=7-5&_nc_sid=bf7eb4&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLnd3dy4xMDc4LkMzIn0%3D&_nc_ohc=0qZtmxR-qC8Q7kNvwGdSXCH&_nc_oc=AdoC3OVFbCGRXoIPAkrtzxjFtK_EIzLAw7SfAgu4t-sHRvEdNeg_zraiK8kEYekL0494VA8kpSNeey7RwjtRkltu&_nc_zt=24&_nc_ht=scontent.cdninstagram.com&edm=AP4hL3IEAAAA&_nc_tpa=Q5bMBQHA4quJEndv5v9Lf9hamFljUET3PYN132f1aQkaaP6SWtjFM9U4BSIzcvaZf9j8StdcoHCwShLg-Q&oh=00_Af3j0PISqyEOzeKAcpuQL0kV_RUqtrWMprhQkX5BZohXQA&oe=69E894E5",
    "followers_count": 1251,
    "follows_count": 1074,
    "media_count": 251
}
App Scoped User ID: 26139958472366508
Instagram Business Account User ID: 17841400550881709


# Test API Which we used to fetch comments details with IG API with Facebook Login

In [8]:
business_discovey_parameters = "thumbnail_url,media_type,media_product_type,timestamp,like_count,comments_count,media_url,permalink"
other_parameters = "comments{id,from,text,timestamp,like_count,media,parent_id,replies{from,timestamp,like_count,text},hidden,user,username}"
fields = business_discovey_parameters + "," + other_parameters
fields

'thumbnail_url,media_type,media_product_type,timestamp,like_count,comments_count,media_url,permalink,comments{id,from,text,timestamp,like_count,media,parent_id,replies{from,timestamp,like_count,text},hidden,user,username}'

In [9]:
url = f"https://graph.instagram.com/v25.0/{instagram_business_account_id}/media"
payload = {
    "fields": fields,
    "access_token": long_access_token
}
response = requests.get(url, params=payload)
data = response.json()
print(json.dumps(data, indent=4))

{
    "data": [
        {
            "media_type": "CAROUSEL_ALBUM",
            "media_product_type": "FEED",
            "timestamp": "2025-02-19T22:13:21+0000",
            "like_count": 66,
            "comments_count": 7,
            "media_url": "https://scontent.cdninstagram.com/v/t51.75761-15/480412422_18492296773018363_2298370280779625650_n.webp?stp=dst-jpg_e35_tt6&_nc_cat=100&ccb=7-5&_nc_sid=18de74&efg=eyJlZmdfdGFnIjoiQ0FST1VTRUxfSVRFTS5iZXN0X2ltYWdlX3VybGdlbi5DMyJ9&_nc_ohc=i1pCpM3_jl8Q7kNvwEjKeMA&_nc_oc=Adok4BhI-3Em4YhgwWV6ssLFdJhQEydJoNGjWF9onGKpLk5FDuuOLeP5eQhRLsSzVSsGx54HdFHMg8Y7GrptThCL&_nc_zt=23&_nc_ht=scontent.cdninstagram.com&edm=ANo9K5cEAAAA&_nc_gid=LMCWTm7x-8cypiIk-FutAQ&_nc_tpa=Q5bMBQFyPGf2vzY5F-JeOgUZZ91gdDmjMw5mHAAoGloAyBS--oMYBEpczcmdYqJ-V6s8M7Nl0nJO2t9x0w&oh=00_Af2zr2TiqLsmIx348hCISKc0zETBPHhykXpmTvR397fDjA&oe=69E88CEE",
            "permalink": "https://www.instagram.com/p/DGRU909S-lj/",
            "comments": {
                "data": [
                    

# Get User Information

In [ ]:
fields = "name,profile_pic,username,follower_count,is_business_follow_user,is_user_follow_business,is_verified_user"

url = f"https://graph.instagram.com/v25.0/1322582323265295"
payload = {
    "fields": fields,
    "access_token": long_access_token
}
response = requests.get(url, params=payload)
data = response.json()
print(json.dumps(data, indent=4))

{
    "error": {
        "message": "User consent is required to access user profile",
        "type": "IGApiException",
        "code": 230,
        "fbtrace_id": "Ac5FGDVc45PH4Dyq15hpGN_"
    }
}
